# Spotify Song Recommender (Regression)

### Load the candidate songs

In [1]:
import duckdb
import numpy as np
import pandas as pd
from datasets import load_dataset

ds = load_dataset("GildasLeDrogoff/spotify-huge-track-analysis-dataset", split="train")
tbl = ds.data.table  # memory-mapped Arrow, nothing loaded into RAM yet

# Same features as the popularity notebook, plus track_popularity (now a feature, not the target).
# No artist columns: they're identical for every song by an artist, so one artist flooded the top 20
FEATURES = """
    any_value(track_name) AS track_name,
    string_agg(artist_name, ', ') AS artists,
    any_value(track_popularity) AS track_popularity,
    any_value(duration_ms) AS duration_ms,
    any_value(explicit)::INT AS explicit,
    any_value(mode) AS mode,
    any_value(track_number) AS track_number,
    year(any_value(album_release_date)) AS release_year,
    any_value(tempo) AS tempo,
    any_value(danceability) AS danceability,
    any_value(energy) AS energy,
    any_value(loudness) AS loudness,
    any_value(speechiness) AS speechiness,
    any_value(acousticness) AS acousticness,
    any_value(instrumentalness) AS instrumentalness,
    any_value(liveness) AS liveness,
    any_value(valence) AS valence,
    any_value(energy_danceability_score) AS energy_danceability_score
"""

pool = duckdb.sql(f"""
    SELECT track_id, {FEATURES}
    FROM tbl TABLESAMPLE 500000 ROWS
    GROUP BY track_id
""").df()
print(len(pool))

/home/darkwarro/Documents/Web_Apps/Academy_Exercises/Spotify_Recommender/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


497315


### Load liked songs (Exportify CSV)

**Before running the next cell, bring your own songs:**

1. Go to [exportify.net](https://exportify.net) and log in with your Spotify account.
2. Export your **Liked Songs** (or any playlist you want recommendations for) as a CSV.
3. Rename the file to `Liked_Songs.csv`.
4. Drag and drop it into the file browser in the left sidebar, below the notebooks in the `Spotify_Recommender` folder.

The file stays in your own workspace, so you only need to do this once.

In [2]:
import pyarrow as pa
import pyarrow.compute as pc

all_liked_csv = pd.read_csv("Liked_Songs.csv", encoding="utf-8-sig")  # export from exportify.net
all_liked_ids = all_liked_csv["Track URI"].str.split(":").str[-1]  # everything you've heard, excluded later

# Train on your taste right now: the 500 songs you liked most recently
liked_csv = all_liked_csv.sort_values("Added At", ascending=False).head(500)
liked_ids = liked_csv["Track URI"].str.split(":").str[-1].drop_duplicates()

# Look them up in the full 56M-row dataset, reading only the track_id column to find matches
# (a plain SQL WHERE over the whole table reads every column and ran out of RAM)
liked_rows = tbl.filter(pc.is_in(tbl.column("track_id"), value_set=pa.array(liked_ids)))

liked = duckdb.sql(f"""
    SELECT track_id, {FEATURES}
    FROM liked_rows
    GROUP BY track_id
""").df()
print(f"Found {len(liked)} of {len(liked_ids)} liked songs, skipped {len(liked_ids) - len(liked)}")

Found 426 of 500 liked songs, skipped 74


### Training data: liked = 1, random songs = 0

In [3]:
# Same song can have several track IDs (album, single, deluxe...), so also match on name + artist
def name_artist_keys(names, artists):
    pairs = pd.DataFrame({"name": names.str.lower().str.strip(), "artist": artists.str.lower().str.split(", ")})
    return pairs.explode("artist").apply(tuple, axis=1)

liked_keys = set(name_artist_keys(all_liked_csv["Track Name"], all_liked_csv["Artist Name(s)"]))
same_song = name_artist_keys(pool["track_name"], pool["artists"]).isin(liked_keys).groupby(level=0).any()

candidates = pool[~pool["track_id"].isin(all_liked_ids) & ~same_song]  # not in liked songs = never listened
print("Dropped as same name + artist:", (same_song & ~pool["track_id"].isin(all_liked_ids)).sum())

# Stratify on popularity: buckets are the deciles of YOUR liked songs' popularity (10% of them each).
# Per bucket, pick 10 random songs for every liked song in it, so popularity alone can't tell them apart
_, bins = pd.qcut(liked["track_popularity"], 10, retbins=True, duplicates="drop")
bins[0], bins[-1] = -np.inf, np.inf  # outermost buckets catch everything below/above
liked_bucket = pd.cut(liked["track_popularity"], bins)
cand_bucket = pd.cut(candidates["track_popularity"], bins)

negatives = pd.concat([
    candidates[cand_bucket == b].sample(n=min(10 * n, (cand_bucket == b).sum()), random_state=42)
    for b, n in liked_bucket.value_counts().items()
])

data = pd.concat([liked.assign(liked=1), negatives.assign(liked=0)], ignore_index=True)
pd.crosstab(pd.cut(data["track_popularity"], bins), data["liked"])

Dropped as same name + artist: 141


liked,0,1
track_popularity,,
"(-inf, 19.0]",440,44
"(19.0, 30.0]",420,42
"(30.0, 39.0]",440,44
"(39.0, 45.0]",420,42
"(45.0, 51.0]",420,42
"(51.0, 57.0]",430,43
"(57.0, 63.0]",500,50
"(63.0, 67.0]",167,38
"(67.0, 73.5]",131,38


In [4]:
id_cols = ["track_id", "track_name", "artists"]

data.drop(columns=id_cols).corr()["liked"].sort_values(ascending=False)

liked                        1.000000
track_popularity             0.115168
explicit                     0.043507
loudness                     0.041990
energy                       0.034437
tempo                        0.025968
energy_danceability_score    0.020562
duration_ms                  0.012456
danceability                -0.001766
release_year                -0.001974
liveness                    -0.009757
track_number                -0.010277
acousticness                -0.016210
speechiness                 -0.019283
instrumentalness            -0.021084
valence                     -0.070359
mode                        -0.075681
Name: liked, dtype: float64

### Split: hide 20% of the songs

In [5]:
from zlib import crc32
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

def test_set_check(identifier, test_ratio):
    return crc32(identifier.encode()) & 0xffffffff < test_ratio * 2**32

def split_train_test_by_id(data, test_ratio, id_column):
    in_test_set = data[id_column].apply(lambda id_: test_set_check(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

train_set, test_set = split_train_test_by_id(data, 0.2, "track_id")

train_labels = train_set["liked"].copy()
train_num = train_set.drop(columns=id_cols + ["liked"])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("minmax_scaler", MinMaxScaler()),
])
train_prepared = num_pipeline.fit_transform(train_num)  # fit on training data only

print("Hidden liked songs in test set:", test_set["liked"].sum(), "of", len(test_set))

Hidden liked songs in test set: 88 of 762


### Compare models (cross-validated RMSE)

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

models = {
    "Linear regression": LinearRegression(),
    "Decision tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random forest": RandomForestRegressor(n_estimators=50, min_samples_leaf=5, max_features=0.5,
                                           n_jobs=-1, random_state=42),
}

print("Baseline RMSE (always predict the average):", train_labels.std())
for name, model in models.items():
    scores = cross_val_score(model, train_prepared, train_labels, scoring="neg_mean_squared_error", cv=5)
    print(f"{name}: CV RMSE {np.sqrt(-scores).mean():.4f}")

Baseline RMSE (always predict the average): 0.3121279885122568
Linear regression: CV RMSE 0.2748
Decision tree: CV RMSE 0.6251
Random forest: CV RMSE 0.5046


### Quiz: how many hidden liked songs land in the model's top picks

In [7]:
test_prepared = num_pipeline.transform(test_set.drop(columns=id_cols + ["liked"]))
K = test_set["liked"].sum()  # as many picks as there are hidden liked songs

print(f"Hidden liked songs: {test_set['liked'].sum()} of {len(test_set)} test songs")
print(f"Random picks: {K * test_set['liked'].mean():.0f} / {K}")
for name, model in models.items():
    model.fit(train_prepared, train_labels)
    top = test_set.assign(score=model.predict(test_prepared)).nlargest(K, "score")
    print(f"{name}: {top['liked'].sum()} / {K}")

Hidden liked songs: 88 of 762 test songs
Random picks: 10 / 88
Linear regression: 21 / 88
Decision tree: 43 / 88
Random forest: 32 / 88


### Your top 20

In [8]:
from sklearn.base import clone

best_model = models["Random forest"]  # swap for whichever won above

# Retrain on all the data, then score every candidate song not used for training
final_model = Pipeline([("prep", clone(num_pipeline)), ("model", clone(best_model))])
final_model.fit(data.drop(columns=id_cols + ["liked"]), data["liked"])

to_rank = candidates.drop(negatives.index)
to_rank = to_rank.assign(score=final_model.predict(to_rank.drop(columns=id_cols)))
to_rank.nlargest(20, "score")[["track_name", "artists", "track_popularity", "score"]]

,track_name,artists,track_popularity,score
138833,Ready 2 Go - Hardwell Remix,Martin Solveig,13,0.567997
96369,Rasteira Tecno,Capoeira Experience,1,0.559291
94402,In The Streets,Joyce Cooling,5,0.556838
392591,Zenith (ASOT 739),Rank 1,1,0.546181
52702,I'd Rather Be Me (With You),Edwin Rhodes,45,0.544123
277216,Tauchenichts,Heinz Erhardt,16,0.541813
433120,The Blue Channel (Live Acoustic TAYF10),Taking Back Sunday,11,0.540482
452528,Blow the speakers 2007,The Moon,7,0.538867
166939,Wir holen die Meisterschaft,FanChants: Hertha Fans,8,0.537143
370940,Do Your Thing,Ratcliffe,7,0.536704
